In [5]:
import os
import random
import numpy as np
import pandas as pd

# ============================================================
# CONFIG
# ============================================================
PACKET_SIZE_BITS = 1500 * 8  # 12000 bits per packet
SCALE_TARGET     = 5.10      # Mbps
MIN_THROUGHPUT   = 0.1       # Mbps
SMOOTH_WINDOW    = 3


# ============================================================
# FUNCTIONS
# ============================================================

def convert_timestamps_to_throughput(timestamps_ms: list[float]) -> list[float]:
    """
    Convert list of packet timestamps (ms) → throughput per detik (Mbps).
    
    Args:
        timestamps_ms: list timestamp dalam milidetik
    
    Returns:
        list throughput mentah (Mbps), satu nilai per detik
    """
    throughput_mbps = []
    current_sec  = 0
    packet_count = 0

    for ts in timestamps_ms:
        sec = int(ts / 1000)

        while current_sec < sec:
            mbps = (packet_count * PACKET_SIZE_BITS) / 1_000_000
            throughput_mbps.append(mbps)
            packet_count = 0
            current_sec += 1

        packet_count += 1

    # flush detik terakhir
    throughput_mbps.append((packet_count * PACKET_SIZE_BITS) / 1_000_000)

    return throughput_mbps


def scale_throughput(
    throughput_mbps : list[float],
    scale_target    : float = SCALE_TARGET,
    min_throughput  : float = MIN_THROUGHPUT
) -> list[float]:
    """
    Scale throughput agar peak == scale_target.
    
    Args:
        throughput_mbps : list throughput mentah (Mbps)
        scale_target    : target peak throughput (Mbps)
        min_throughput  : nilai minimum setelah scaling
    
    Returns:
        list throughput setelah scaling
    """
    max_tp       = max(throughput_mbps) if throughput_mbps else 1.0
    scale_factor = (scale_target / max_tp) if max_tp > 0 else 1.0

    return [max(min_throughput, tp * scale_factor) for tp in throughput_mbps]


def smooth_throughput(
    throughput_mbps : list[float],
    window          : int = SMOOTH_WINDOW
) -> list[float]:
    """
    Haluskan throughput dengan rolling mean.
    
    Args:
        throughput_mbps : list throughput (Mbps)
        window          : ukuran window rolling mean
    
    Returns:
        list throughput setelah smoothing
    """
    return (
        pd.Series(throughput_mbps)
        .rolling(window=window, min_periods=1)
        .mean()
        .tolist()
    )


def load_trace_file(filepath: str) -> dict | None:
    """
    Baca satu file trace Mahimahi (.log) → dict {name, data}.
    
    Args:
        filepath: path ke file .log
    
    Returns:
        dict {"name": filename, "data": list throughput smoothed}
        atau None jika gagal
    """
    filename = os.path.basename(filepath)

    try:
        with open(filepath, 'r') as f:
            timestamps_ms = [float(line.strip()) for line in f if line.strip()]

        if not timestamps_ms:
            print(f"⚠️  Skip (kosong): {filename}")
            return None

        raw      = convert_timestamps_to_throughput(timestamps_ms)
        scaled   = scale_throughput(raw)
        smoothed = smooth_throughput(scaled)

        return {"name": filename, "data": smoothed}

    except Exception as e:
        print(f"❌ Gagal memproses {filename}: {e}")
        return None


def load_trace_folder(folder_path: str = "traces_folder/mahimahi") -> list[dict]:
    """
    Baca semua file .log dalam folder → list of dict {name, data}.
    Jika folder kosong/tidak ada, kembalikan synthetic fallback.
    
    Args:
        folder_path: path ke folder berisi file .log
    
    Returns:
        list of dict {"name": ..., "data": [...]}
    """
    traces = []

    if os.path.exists(folder_path):
        files = sorted(f for f in os.listdir(folder_path) if f.endswith('.log'))

        for file in files:
            result = load_trace_file(os.path.join(folder_path, file))
            if result:
                traces.append(result)
                print(
                    f"✅ {file} | "
                    f"peak={max(result['data']):.2f} Mbps | "
                    f"{len(result['data'])} detik"
                )

    # fallback synthetic
    if not traces:
        print("⚠️  Folder tidak ditemukan / kosong. Pakai fallback synthetic.")
        synthetic = [3.0 + 1.5 * np.sin(i * 0.1) for i in range(500)]
        traces.append({"name": "synth", "data": synthetic})

    print(f"\n✅ Total {len(traces)} trace dimuat. Scale target: {SCALE_TARGET:.2f} Mbps")
    return traces


def get_bandwidth(traces: list[dict], trace_idx: int, ptr: int) -> tuple[float, int]:
    """
    Ambil nilai bandwidth berikutnya dari trace yang aktif.
    
    Args:
        traces    : list trace hasil load_trace_folder()
        trace_idx : index trace yang aktif
        ptr       : posisi pointer saat ini
    
    Returns:
        (bandwidth_mbps, ptr_baru)
    """
    data = traces[trace_idx]["data"]
    bw   = data[ptr]
    ptr  = (ptr + 1) % len(data)
    return bw, ptr


def write_mahimahi_trace(
    throughput_mbps_list : list[float],
    output_path          : str,
    bytes_per_pkt        : float = 1500.0
) -> None:
    """
    Generate ulang file Mahimahi trace dari list throughput (Mbps).
    Format output: satu timestamp (ms) per baris.

    Args:
        throughput_mbps_list : list throughput per detik (Mbps)
        output_path          : path file output .log
        bytes_per_pkt        : ukuran paket dalam bytes (default 1500)
    """
    millisec_time = 0

    with open(output_path, 'w') as f:
        f.write(str(millisec_time) + '\n')

        for throughput_mbps in throughput_mbps_list:

            # Convert Mbps → Bytes/ms
            bytes_per_ms  = (throughput_mbps * 1_000_000) / (8 * 1000)
            pkt_per_ms    = bytes_per_ms / bytes_per_pkt

            pkt_count     = 0
            millisec_count = 0

            while millisec_count < 1000:  # 1 detik = 1000 ms
                millisec_count += 1
                millisec_time  += 1

                to_send = np.floor(millisec_count * pkt_per_ms - pkt_count)

                for _ in range(int(to_send)):
                    f.write(str(millisec_time) + '\n')

                pkt_count += to_send

In [ ]:
# 1. Baca file asli
with open("traces_folder/mahimahi/trace_001.log", "r") as f:
    timestamps_ms = [float(line.strip()) for line in f if line.strip()]

# 2. Convert → scale → smooth
raw      = convert_timestamps_to_throughput(timestamps_ms)
scaled   = scale_throughput(raw)
smoothed = smooth_throughput(scaled)

# 3. Tulis ulang jadi file Mahimahi baru
write_mahimahi_trace(smoothed, "output/trace_001_scaled.log")

print("✅ File trace baru berhasil ditulis")

✅ report_bicycle_0001.log | peak=4.50 Mbps | 531 detik
✅ report_bicycle_0002.log | peak=4.90 Mbps | 658 detik
✅ report_bus_0001.log | peak=4.68 Mbps | 607 detik
✅ report_bus_0002.log | peak=4.48 Mbps | 546 detik
✅ report_bus_0003.log | peak=4.75 Mbps | 763 detik
✅ report_bus_0004.log | peak=4.91 Mbps | 281 detik
✅ report_bus_0005.log | peak=4.98 Mbps | 338 detik
✅ report_bus_0006.log | peak=4.98 Mbps | 516 detik
✅ report_bus_0007.log | peak=4.83 Mbps | 483 detik
✅ report_bus_0008.log | peak=4.83 Mbps | 346 detik
✅ report_bus_0009.log | peak=4.67 Mbps | 544 detik
✅ report_bus_0010.log | peak=4.75 Mbps | 442 detik
✅ report_bus_0011.log | peak=4.25 Mbps | 232 detik
✅ report_car_0001.log | peak=4.82 Mbps | 468 detik
✅ report_car_0002.log | peak=4.78 Mbps | 566 detik
✅ report_car_0003.log | peak=4.57 Mbps | 495 detik
✅ report_car_0004.log | peak=4.73 Mbps | 581 detik
✅ report_car_0005.log | peak=4.95 Mbps | 454 detik
✅ report_car_0006.log | peak=4.74 Mbps | 419 detik
✅ report_car_0007.log |

In [ ]:
import os

input_folder  = "./mahimahi_traces/"
output_folder = "scaled_traces"
os.makedirs(output_folder, exist_ok=True)

for filename in sorted(os.listdir(input_folder)):
    if not filename.endswith(".log"):
        continue

    # Baca
    with open(os.path.join(input_folder, filename), "r") as f:
        timestamps_ms = [float(line.strip()) for line in f if line.strip()]

    # Convert
    raw      = convert_timestamps_to_throughput(timestamps_ms)
    scaled   = scale_throughput(raw)
    smoothed = smooth_throughput(scaled)

    # Tulis ulang
    out_path = os.path.join(output_folder, filename)
    write_mahimahi_trace(smoothed, out_path)

    print(f"✅ {filename} → {len(smoothed)} detik | peak={max(smoothed):.2f} Mbps")

✅ report_bicycle_0001.log → 531 detik | peak=4.50 Mbps
✅ report_bicycle_0002.log → 658 detik | peak=4.90 Mbps
✅ report_bus_0001.log → 607 detik | peak=4.68 Mbps
✅ report_bus_0002.log → 546 detik | peak=4.48 Mbps
✅ report_bus_0003.log → 763 detik | peak=4.75 Mbps
✅ report_bus_0004.log → 281 detik | peak=4.91 Mbps
✅ report_bus_0005.log → 338 detik | peak=4.98 Mbps
✅ report_bus_0006.log → 516 detik | peak=4.98 Mbps
✅ report_bus_0007.log → 483 detik | peak=4.83 Mbps
✅ report_bus_0008.log → 346 detik | peak=4.83 Mbps
✅ report_bus_0009.log → 544 detik | peak=4.67 Mbps
✅ report_bus_0010.log → 442 detik | peak=4.75 Mbps
✅ report_bus_0011.log → 232 detik | peak=4.25 Mbps
✅ report_car_0001.log → 468 detik | peak=4.82 Mbps
✅ report_car_0002.log → 566 detik | peak=4.78 Mbps
✅ report_car_0003.log → 495 detik | peak=4.57 Mbps
✅ report_car_0004.log → 581 detik | peak=4.73 Mbps
✅ report_car_0005.log → 454 detik | peak=4.95 Mbps
✅ report_car_0006.log → 419 detik | peak=4.74 Mbps
✅ report_car_0007.log →